# xbrl-frontier Phase 4: T4 load tests per serving variant (owner-run)

`vllm bench serve` over the pinned latency prompts (`serving.load_test`): warm-up, then closed-loop points at `serving.concurrency_levels.kaggle_t4` and Poisson points at `serving.request_rate_sweep_rps`, `serving.requests_per_point` requests each. The load generator is colocated, so network time is excluded (stated in the table).
**Settings:** GPU T4, Internet on. **Attach:** `xbrl-frontier-src` (a commit that contains `data/frontier_latency_ids.txt`) and the export dataset(s) with the variants that passed the quantization gates.

In [ ]:
MODEL = 'Qwen/Qwen3-1.7B'
VARIANTS = {   # variant -> exported checkpoint dir (only variants that passed gates.quantized_*)
    'fp16': '/kaggle/input/xbrl-frontier-export-1p7b-s1/merged-fp16',
    'int8-w8a8': '/kaggle/input/xbrl-frontier-export-1p7b-s1/int8-w8a8',
    'awq-int4': '/kaggle/input/xbrl-frontier-export-1p7b-s1/awq-int4',
}

In [ ]:
import glob, os, shutil, subprocess, sys
dst = '/kaggle/working/xbrl-frontier'
tarballs = glob.glob('/kaggle/input/**/xbrl-frontier-src.tar.gz', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p)))
         for p in glob.glob('/kaggle/input/**/docs/config/thresholds.yaml', recursive=True)]
if tarballs:
    assert len(tarballs) == 1, tarballs
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['tar', '-xzf', tarballs[0], '-C', dst], check=True)
else:  # Kaggle auto-extracted the upload
    assert len(trees) == 1, trees
    shutil.copytree(trees[0], dst, dirs_exist_ok=True)
os.chdir(dst)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import yaml
pins = yaml.safe_load(open('docs/config/thresholds.yaml'))['pins']['software']
missing = [k for k in ('vllm',) if not pins.get(k)]
assert not missing, f'pins.software empty for {missing}: owner approval needed first'
print(pins)

In [ ]:
!pip install -q "vllm=={pins['vllm']}" "pyyaml==6.0.3" "pyarrow==25.0.1"
!pip uninstall -y -q torchaudio  # Kaggle's torchaudio is built for another CUDA than vllm's torch; transformers imports it
import importlib.metadata as md
keys = ('vllm',)
installed = {k: md.version(k) for k in keys}
bad = {k: (v, pins[k]) for k, v in installed.items() if v != pins[k]}
assert not bad, f'installed != pins.software: {bad}; restart the kernel and run all again'
import importlib
imported = {k: importlib.import_module(k).__version__ for k in keys}
bad = {k: (v, pins[k]) for k, v in imported.items() if v != pins[k]}
assert not bad, f'imported != pins.software: {bad}; restart the kernel (Run > Restart) and run all again'
import torch; print(imported, 'torch', torch.__version__, 'CUDA', torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
out = f'/kaggle/working/load-kaggle_t4-{MODEL.split("/")[-1]}'
for variant, ckpt in VARIANTS.items():
    !python -m serving.load_test --card kaggle_t4 --model {MODEL} --variant {variant} --checkpoint {ckpt} --out {out}
!ls -la {out}